# Unit 4 — Feature Extraction from Signals

**Syllabus coverage:** Unit 4 — Feature Extraction from Signals

**Experiments covered:** #7 (STFT Spectrogram), #8 (Bandpass Filtering), #9 (Signal Denoising)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pywt
from pathlib import Path
from scipy import signal as sp_signal

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')

REPO_ROOT = Path('.').resolve().parent
DATA_DIR = REPO_ROOT / 'data' / 'processed'
FIG_DIR = REPO_ROOT / 'outputs' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
SAVEKW = dict(dpi=150, bbox_inches='tight')

## Section 1 — STFT Spectrogram (Experiment #7)

In [ ]:
df = pd.read_csv(DATA_DIR / 'entropy_signal.csv', parse_dates=['meet_date'])
H = df['entropy_H'].dropna().values
seq_ids = df['race_seq_id'].values
dates = pd.to_datetime(df['meet_date'])
N = len(H)
print(f'Signal length: {N}')

# STFT
nperseg = 128
noverlap = 96
f_stft, t_stft, Zxx = sp_signal.stft(H, fs=1.0, window='hann',
                                       nperseg=nperseg, noverlap=noverlap)

power_db = 10 * np.log10(np.abs(Zxx) ** 2 + 1e-12)

fig, ax = plt.subplots(figsize=(16, 6))
im = ax.pcolormesh(t_stft, f_stft, power_db, cmap='viridis', shading='auto')
ax.set_ylabel('Frequency (cycles/race)')
ax.set_title('STFT Spectrogram of Entropy Signal')
fig.colorbar(im, ax=ax, label='Power (dB)')

# Annotate x-axis with approximate years
year_positions = []
year_labels = []
for year in sorted(dates.dt.year.unique()):
    first_idx = df[dates.dt.year == year].index[0]
    if first_idx < len(seq_ids):
        year_positions.append(seq_ids[first_idx])
        year_labels.append(str(year))

ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())
ax2.set_xticks(year_positions[::2])
ax2.set_xticklabels(year_labels[::2], fontsize=8)
ax2.set_xlabel('Year')

ax.set_xlabel('Race Position')
fig.savefig(FIG_DIR / 'unit4_stft_spectrogram.png', **SAVEKW)
plt.show()

## Section 2 — Bandpass Filtering (Experiment #8)

In [ ]:
# Butterworth bandpass filter design
fs = 1.0  # 1 race per sample

# Low band: seasonal variation (1/200 to 1/50 cycles/race)
low_band = [1/200, 1/50]
b_low, a_low = sp_signal.butter(4, low_band, btype='band', fs=fs)
H_low = sp_signal.filtfilt(b_low, a_low, H)

# High band: race-to-race noise (1/10 to 1/2 cycles/race = Nyquist)
high_band = [1/10, 0.49]
b_high, a_high = sp_signal.butter(4, high_band, btype='band', fs=fs)
H_high = sp_signal.filtfilt(b_high, a_high, H)

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
axes[0].plot(seq_ids, H, lw=0.3, alpha=0.6, color='gray', label='Original')
axes[0].plot(seq_ids, H_low, lw=1.5, color='steelblue', label='Low band (seasonal)')
axes[0].set_title('Original + Seasonal Component')
axes[0].legend(loc='upper right')
axes[0].set_ylabel('H (bits)')

axes[1].plot(seq_ids, H_high, lw=0.3, color='coral', alpha=0.7)
axes[1].set_title('High Band (Race-to-Race Noise)')
axes[1].set_ylabel('H (bits)')

# Residual
residual = H - H_low - H_high
axes[2].plot(seq_ids, residual, lw=0.3, color='green', alpha=0.7)
axes[2].set_title('Residual (mid-frequency)')
axes[2].set_xlabel('Race Sequence ID')
axes[2].set_ylabel('H (bits)')

fig.tight_layout()
fig.savefig(FIG_DIR / 'unit4_bandpass_filtering.png', **SAVEKW)
plt.show()

# Variance captured by each band
var_total = np.var(H)
var_low = np.var(H_low)
var_high = np.var(H_high)
print(f'Total variance        : {var_total:.6f}')
print(f'Low band variance     : {var_low:.6f} ({var_low/var_total:.1%})')
print(f'High band variance    : {var_high:.6f} ({var_high/var_total:.1%})')

## Section 3 — Signal Denoising (Experiment #9)

In [ ]:
# Inject synthetic Gaussian noise (SNR = 10 dB)
snr_db = 10
signal_power = np.mean(H ** 2)
noise_power = signal_power / (10 ** (snr_db / 10))
rng = np.random.default_rng(42)
noise = rng.normal(0, np.sqrt(noise_power), len(H))
H_noisy = H + noise

print(f'Original signal power: {signal_power:.4f}')
print(f'Added noise power: {noise_power:.4f}')
print(f'Target SNR: {snr_db} dB')

In [ ]:
# Denoising methods
# a) Moving average (window=15)
kernel = np.ones(15) / 15
H_ma = sp_signal.convolve(H_noisy, kernel, mode='same')

# b) Median filter
H_med = sp_signal.medfilt(H_noisy, kernel_size=15)

# c) Wavelet soft-thresholding
wavelet = 'db4'
coeffs = pywt.wavedec(H_noisy, wavelet, level=5)
# Estimate noise sigma from finest detail coefficients
sigma = np.median(np.abs(coeffs[-1])) / 0.6745
threshold = sigma * np.sqrt(2 * np.log(len(H_noisy)))
coeffs_thresh = [coeffs[0]]  # Keep approximation
for c in coeffs[1:]:
    coeffs_thresh.append(pywt.threshold(c, value=threshold, mode='soft'))
H_wavelet = pywt.waverec(coeffs_thresh, wavelet)[:len(H)]

# Compute MSE and SNR improvement
def mse(a, b):
    """Mean squared error."""
    n = min(len(a), len(b))
    return np.mean((a[:n] - b[:n]) ** 2)

def snr_improvement(original, noisy, denoised):
    """Compute SNR improvement in dB."""
    n = min(len(original), len(denoised))
    mse_noisy = np.mean((original[:n] - noisy[:n]) ** 2)
    mse_denoised = np.mean((original[:n] - denoised[:n]) ** 2)
    if mse_denoised == 0:
        return float('inf')
    return 10 * np.log10(mse_noisy / mse_denoised)

methods = [
    ('Moving Average (k=15)', H_ma),
    ('Median Filter (k=15)', H_med),
    ('Wavelet Soft-Threshold', H_wavelet),
]

print(f'{"Method":30s}  {"MSE":>10s}  {"SNR Improvement (dB)":>20s}')
print('-' * 65)
for name, denoised in methods:
    m = mse(H, denoised)
    s = snr_improvement(H, H_noisy, denoised)
    print(f'{name:30s}  {m:10.6f}  {s:20.3f}')

In [ ]:
# Visual comparison
fig, axes = plt.subplots(5, 1, figsize=(16, 15), sharex=True)
rng_show = slice(500, 800)  # Zoom into a segment
x = seq_ids[rng_show]

axes[0].plot(x, H[rng_show], lw=1, color='steelblue')
axes[0].set_title('Original Signal')

axes[1].plot(x, H_noisy[rng_show], lw=0.5, color='gray')
axes[1].set_title('Noisy Signal (SNR = 10 dB)')

for i, (name, denoised) in enumerate(methods):
    axes[i + 2].plot(x, H[rng_show], lw=0.8, alpha=0.4, color='steelblue', label='Original')
    axes[i + 2].plot(x, denoised[rng_show], lw=1.2, color='coral', label=name)
    axes[i + 2].set_title(name)
    axes[i + 2].legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('Race Sequence ID')
for ax in axes:
    ax.set_ylabel('H (bits)')

fig.tight_layout()
fig.savefig(FIG_DIR / 'unit4_denoising_comparison.png', **SAVEKW)
plt.show()

In [ ]:
# Apply wavelet denoising to the REAL signal (not synthetic noise)
coeffs_real = pywt.wavedec(H, 'db4', level=5)
sigma_real = np.median(np.abs(coeffs_real[-1])) / 0.6745
thresh_real = sigma_real * np.sqrt(2 * np.log(len(H)))
coeffs_real_t = [coeffs_real[0]] + [pywt.threshold(c, value=thresh_real, mode='soft') for c in coeffs_real[1:]]
H_denoised_real = pywt.waverec(coeffs_real_t, 'db4')[:len(H)]

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(seq_ids, H, lw=0.3, alpha=0.4, color='gray', label='Original')
ax.plot(seq_ids, H_denoised_real, lw=1, color='steelblue', label='Wavelet-denoised')
ax.set_xlabel('Race Sequence ID')
ax.set_ylabel('Entropy H (bits)')
ax.set_title('Wavelet Denoising Applied to Real Entropy Signal')
ax.legend()
fig.savefig(FIG_DIR / 'unit4_wavelet_denoised_real.png', **SAVEKW)
plt.show()

## Summary

- The STFT spectrogram shows time-varying frequency content in the entropy signal.
- Bandpass filtering separates seasonal trends from race-to-race fluctuations.
- Wavelet soft-thresholding achieves the best denoising performance among the three methods tested.
- Applied to the real signal, wavelet denoising reveals the underlying smooth structure.